# Tutorial: Calibration of cross section models

---
_Developed by D. Brakenhoff and M. Bakker_

This tutorial describes how to build and calibrate steady and transient cross-section models with Timflow.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import timflow as tf

## Example problem: river next to a polder

The example we will be working on is shown in the figure below. There is a river,
with a width of about 300 m and a mean water level of around NAP+0.7 m. A dike protects
a low-lying area situated to the right of our river. The water level in this polder is
controlled at NAP-1.3 m. The red dashed line indicates the head in the aquifer. This is what we want to compute using Timflow. 

<img src="xsec_schematic.png" alt="Cross-section" width="800">

### Subsurface

The subsurface consists of a low permeable layer (clay) and underneath an aquifer with
a total thickness of ca. 25 m. The bottom of the aquifer can be considered impermeable
for our problem. The summer bed of the river cuts into the leaky layer, and we know
there is strong hydraulic connection between the river and the aquifer.

| Zone | Layer | Top [mNAP]| Bottom [mNAP] | $k_h$ [m/d]| c [d] |
|:---- |:----- | ---------:| -------------:| ----------:| -----:|
|river | leaky layer| -4.0 |           -5.0|            |    1  |
|polder| leaky layer|  -1.0|           -5.0|            | 1000  |
|      | aquifer|      -5.0|          -30.0|       10.0 |       |

### Goal(s) of this tutorial

Our final goal is to setup a calibrated cross-section model in Timflow that is able to
match the observed heads that is measured in the observation wells shown above.

We will run through a number of smaller steps to build up to get to that final goal.

1. Build a steady-state cross-section model.
2. Build a transient cross-section model.
3. Setup a calibration with both models.
4. Run calibration.
5. Inspect the results.

## 1. Build a steady-state cross-section model

To start a cross-section model, we create a Model class. This is the container to which
we will add all our model elements. For cross-section models we can use `mlss =
tf.steady.ModelXsection()`. The only information we need to supply is the number of
aquifers in our model, which is 1.

Next, we need to add the river and the polder zones to our
cross-section model. We will add two `XsectionMaq` inhomogeneities with a single
aquifer. We will assume our river extends to infinity towards the left of our model. The
polder starts at x=0 m and extends towards positive infinity.

We need to choose initial values for the following geohydrological parameters to build
our model. Pick the values above, or what you think might be realistic values. (We will
calibrate those later!). We need to specify the following parameters:

- `kaq`: hydraulic conductivity aquifer
- `c0` in the river zone: the bed resistance of the river
- `c0` in the polder: the resistance of leaky layer

In [ ]:
# setup model parameters

z_riv = [-4.0, -5.0, -30]  # elevations layers in river section, m NAP
kaq = 10.0  # hydraulic conductivity aquifer, m/d
c_riv = 1.0  # bed resistance river, d

hstar_river = 0.75  # mean water level in the river, m NAP

In [ ]:
mlss = tf.steady.ModelXsection(naq=1)
riv_ss = tf.steady.XsectionMaq(
    mlss,
    x1=-np.inf,
    x2=0.0,
    z=z_riv,
    kaq=kaq,
    c=c_riv,
    hstar=hstar_river,
    topboundary="semi",
    name="river",
)

**Question 1** Add an `XsectionMaq` to the model representing the polder. Don't forget
to set `hstar` to the polder water level . It is also recommended to name your
`XsectionMaq`'s by supplying a name.

In [ ]:
# your code here

Solve the model with `mlss.solve()`.

In [ ]:
mlss.solve()

**Question 2** Plot a cross section of our model with `mlss.plots.xsection()`. Check out the docstring
with `mlss.plots.xsection?` to customize your cross-section plot.

In [ ]:
# your code here

Next plot the head in the aquifer using `mlss.plots.head_along_line(xmin, xmax, ymin,
ymax, npts)`. Additionally, plot the mean observed heads at the observation wells. You
can use the `sstart` argument to select the starting value of the x-axis, so that the
coordinates match those of your model.

In [ ]:
ax = mlss.plots.head_along_line(-100, 500, 0, 0, 101, figsize=(10, 2.5), sstart=-100)

ax.legend(loc=(0, 1), ncol=5, frameon=False)
ax.set_ylabel("m NAP")
ax.set_xlabel("x (m)");

## 2. Build a transient cross-section model

The next step is to build a transient cross-section model. But before we do that, we
need a time series of the river level. Load the river time series and plot the data.

In [ ]:
river = pd.read_csv("data/river.csv", index_col=[0], parse_dates=True).squeeze()
river.plot(figsize=(10, 2));

Now we can start building our transient cross-section using
`tf.transient.ModelXsection()`. We can use the steady model as our template, but we
need to add a bit more information for transient computations.

- We need to supply a `tmin` and `tmax` to our model class so that timflow knows for
which time period we want to compute the heads. Let's use a very small `tmin=1e-5` and
a `tmax=100` days.

- We also need to specify the storage coefficients of the aquifer (`Saq`). Use a value
of `1e-4` as your initial guess.

- Finally, we need to add the river stage time series to the river `XsectionMaq`
through the keyword argument `tsandhstar`. We do this by supplying a list of time and
stage pairs `[(t0, h0), (t1, h1), (tn, hn)]`, where `ti` is the time in days and `hi`
is the mean river water level in each period. For more information on why we need to
compute the mean level for each period, see [this how-to
guide](https://timflow.readthedocs.io/en/latest/transient/00userguide/howtos/howto_fluctuating_head_boundary.html)
on the Timflow documentation website.

In [ ]:
mltr = tf.transient.ModelXsection(naq=1, tmin=1e-5, tmax=1e2)

Compute the mean river stage for each time step starting at $t_i$. Recall
`timflow.transient` only computes head changes, so we are computing the changes in
river level relative to the mean river stage.

In [ ]:
# time in days, removing last observation
triv = ((river.index - river.index[0]).total_seconds() / (3600 * 24))[:-1]

# compute the stage change relative to the mean river stage
hriv = river.values - hstar_river

# compute the mean stage for each time step
hriv_mean = (hriv[1:] + hriv[:-1]) / 2

# create a list of time and mean stage pairs
tsandhstar = list(zip(triv, hriv_mean))

# view first few entries of tsandhstar
tsandhstar[:5]

Now we can add the `XsectionMaq` representing the river.

In [ ]:
Saq = 1e-4  # specific storage aquifer, 1/m

riv_tr = tf.transient.XsectionMaq(
    mltr,
    x1=-np.inf,
    x2=0.0,
    z=z_riv,
    kaq=kaq,
    c=c_riv,
    Saq=Saq,
    tsandhstar=tsandhstar,
    topboundary="semi",
    name="river",
)

**Question 3** Add the polder `XsectionMaq` to the model. Don't forget to supply the
specific storage and name your element. There is no need to specify `tsandhstar` in the
polder, the polder level remains constant during our simulation. After you have added
the polder, solve the model with `mltr.solve()`.

In [ ]:
# your code here

**Question 4** Plot the head in the aquifer from x=-100 to x=500 m at t = 1.5, 1.625
and 2 days after the start of the simulation. These times correspond to low tide, high
tide and the next low tide on the river (the tidal signal is not symmetric). Use
`mltr.plots.head_along_line()` to create your plot. Bonus points for also plotting the
river stage at those times in the plot.

In [ ]:
# your code here

We now have the average groundwater flow from the river towards the polder, which is
represented by our steady model. And we have the change in head caused by the tidal
fluctuations in our river.

Since we are doing analytic element modeling we can combine these two results by adding
them together to compute the head at any location at any time relative to NAP.

Let's compute the head in m NAP at $x=0$.

In [ ]:
x = 0.0
t = np.linspace(0, 14, 300)

h_steady = mlss.head(x, 0)
h_transient = mltr.head(x, 0, t)[0]

plt.figure(figsize=(10, 2.5))
plt.plot(t, h_steady + h_transient, label=f"model at x={x}", color="C0")
plt.legend(loc=(0, 1), ncol=5, frameon=False)
plt.xlabel("time (days)")
plt.ylabel("head (m NAP)")
plt.grid(True)

## 3. Setup calibration

Now we want to calibrate our model on the observed heads. So let's load the observed
heads first.

Observation wells have been placed at different distances along a transect
perpendicular to the river. They all measure the head in the aquifer. The distances at
which each well is placed relative to the top of the dike is given in the table below.

| Observation Well | Distance [m] |
|:---------------- | ------------:|
| obs1             |            0 |
| obs2             |           54 |
| obs3             |          172 |
| obs4             |          194 |


In [ ]:
obs1 = pd.read_csv("data/obs1.csv", index_col=0, parse_dates=True).squeeze()
obs2 = pd.read_csv("data/obs2.csv", index_col=0, parse_dates=True).squeeze()
obs3 = pd.read_csv("data/obs3.csv", index_col=0, parse_dates=True).squeeze()
obs4 = pd.read_csv("data/obs4.csv", index_col=0, parse_dates=True).squeeze()

Plot the time series of the head in the observation wells and the river stage in one figure.

In [ ]:
# Plot the results.
fig, ax = plt.subplots(1, 1, figsize=(10, 3))
river.plot(ax=ax, color="k", label="river")
for iobs, ix in zip([obs1, obs2, obs3, obs4], [0, 54, 172, 194]):
    iobs.plot(ax=ax, label=f"{iobs.name} (x={ix}m)")
ax.legend(loc=(0, 1), ncol=5, frameon=False)
ax.set_ylabel("m NAP")
ax.grid(True)



The next step is to calibrate our models on the observed heads. We want to calibrate on
the observed heads relative to NAP, so we need to calibrate our steady and transient
models simultaneously.

This is done with the `tf.Calibrate()` tool, which accepts both steady and/or transient
models. Let's go ahead and create a `Calibrate()` object with our two models.

In [ ]:
cal = tf.Calibrate(transient_model=mltr, steady_model=mlss)

### Add observed heads

Next, we have to add our observed head time series so we have something to calibrate on.
Adding the head time series is done using `cal.add_head_time_series()`. We have to
supply the following information:

- `name` of the time series
- `x`, `y` location of the observation well
- `layer` in which the well is measuring
- `t` the times in days relative to the starting date of the model
- `h` the observed heads

Let's start by adding the observations from the first observation well in `obs1`.

In [ ]:
t = (obs1.index - river.index[0]).total_seconds() / (3600 * 24)  # compute time in days

# add heads observation well 1 as calibration target
cal.add_head_time_series(obs1.name, x=0.0, y=0.0, layer=0, t=t, h=obs1.values)

### Specify calibration parameters

Now we have to specify which parameters we want to vary during the calibration. We can
specify any of the aquifer parameters (`kaq`, `Saq`, `c`) with
`cal.set_aquifer_parameter()`. When we set the parameters we have to supply the
following information:

- `name`, the name of the parameter, one of `kaq`, `Saq` or `c`.
- `layers`, the layers in which the parameter is set to a specific value
- `initial`, your initial guess (set to None to use the current model value).
- `pmin` and `pmax`, the parameter bounds
- `log_scale`, whether to log-scale the parameter in the calibration

Then there are two more things we need to set. Recall that we set up our two models
using two inhomogeneities representing the river and the polder, and each of these
zones has its own parameters. If we want to use a single value for the hydraulic
conductivity of the entire aquifer (so both underneath the river and the polder) in
both models, we have to tell the `Calibrate` class that.

Setting the calibration parameter so that it varies both the aquifer hydraulic
conductivity across both zones is done with the `inhoms` keyword argument. Pass a list
of the names of the inhomogeneities in order to link the parameter between these zones.

```python
cal.set_aquifer_parameter(name="kaq", ..., inhoms=["river", "polder"])
```

Similarly, linking the parameter across both the steady and transient models, is done
with the `model` keyword argument. Options are `"steady"`, `transient"` or `"both"`.
The latter is the default, so we don't really need to specify it for the hydraulic
conductivity, but we will need to adjust this value when specifying the specific
storage, because this parameter only pertains to the transient model.

```python
cal.set_aquifer_parameter("Saq", ..., model="transient")
```

To get an overview of all the aquifer parameters for a model use `aquifer_summary()`:

In [ ]:
# view all parameters in the transient model
mltr.aquifer_summary()

Now let's pick the set of parameters we want to calibrate:

- `kaq`, the hydraulic conductivity for the aquifer, initial guess 10 m/d, with bounds
between 1 and 100 m/d
- `Saq`, the specific storage coefficient for the aquifer, initial guess 1E-4, with
bounds 1E-7 and 1E-2
- `c`, the resistance of the river bed, initial guess 1 day, with bounds 0.1 and 100
days
- `c`, the resistance of the leaky layer in the polder, initial guess 1000 days and
bounds between 100 and 10,000 days.

Set the parameters

In [ ]:
cal.set_aquifer_parameter(
    "kaq",
    layers=0,
    initial=10.0,
    pmin=1.0,
    pmax=100.0,
    log_scale=True,
    inhoms=["river", "polder"],
    model="both",
)
cal.set_aquifer_parameter(
    "Saq",
    layers=0,
    initial=1e-4,
    pmin=1e-7,
    pmax=1e-2,
    log_scale=True,
    inhoms=["river", "polder"],
    model="transient",
)
cal.set_aquifer_parameter(
    "c",
    layers=0,
    initial=1.0,
    pmin=0.1,
    pmax=100.0,
    log_scale=True,
    inhoms=["river"],
    model="both",
)
cal.set_aquifer_parameter(
    "c",
    layers=0,
    initial=1000.0,
    pmin=10,
    pmax=1e4,
    log_scale=True,
    inhoms=["polder"],
    model="both",
)

## 4. Calibrate the model

Now run the calibration with `cal.lmfit()`.

In [ ]:
cal.lmfit(report=True)

Plot the simulated heads vs the observed heads using `cal.plot_transient_results()`. If all went well you should get a quite good fit on the observed data in observation well 1.

In [ ]:
f, axes = cal.plot_transient_results(figsize=(10, 2))

View the optimal parameters with `cal.parameters`.

_**Note**: the parameter names in the dataframe indicate which layers and which inhomogeneities the parameter applies to: `<param>_<layer_from>_<layer_to>_<inhom_name.1>...`_

In [ ]:
cal.parameters

Check the value of the `kaq` parameter in the river section of the steady model. It should match the optimal value from the table above.

In [ ]:
riv_ss.kaq

**Question 5** Check the value of the `c_0` polder parameter in both the steady and the transient model.

In [ ]:
# your code here

## 5. Inspect the calibration results

In this section we will examine the calibration results.

**Question 6** Run the calibration again with 2 observation wells, then 3 and finally
with all 4 observation wells. What is the RMSE (`cal.rmse()`), and how do the estimated
parameters change?

In [ ]:
# your code here

**Question 7** Plot the calibration results for the final run with all 4 observation wells.

In [ ]:
# your code here

So the fit should be quite good, but there's actually something going on with our
parameters. Let's look at the calibration result `cal.result`, and specifically the
section on correlations.

In [ ]:
cal.result

All our parameter correlations are equal to $\pm 1$! Let's explore this a bit further
in the next question.

_**Note:** we are working on an automatic informative warning when the calibration
results show very high parameter correlations._

**Question 8** Re-run the calibration but now try calibrating fewer parameters. E.g.
only modify `Saq` and `c_0` and `c_polder` and keep the value of `kaq` constant. Try a
few different combinations. What combination of parameters gets you the lowest RMSE?
How does this result compare to the result above? What are the parameter correlations
for your final result?


We have added a helper function below that you can use to get a calibration object with
the two models and all observed heads added. This is so that the initial values of the
parameters always start out at the same initial values. When you run a new calibration
on a model that was already calibrated, the parameter values will have changed and give
you different starting values for a subsequent calibration run.

Usage: `cal = get_calibration_object()`

In [ ]:
def get_calibration_object(kaq=10, c_riv=1, c_polder=1000, Saq=1e-4):
    z_polder = [-1.0, -5.0, -30]
    c_polder = 1_000.0
    mlss = tf.steady.ModelXsection(naq=1)
    tf.steady.XsectionMaq(
        mlss,
        x1=-np.inf,
        x2=0.0,
        z=z_riv,
        kaq=kaq,
        c=c_riv,
        name="river",
        topboundary="semi",
        hstar=hstar_river,
    )
    tf.steady.XsectionMaq(
        mlss,
        x1=0.0,
        x2=np.inf,
        z=z_polder,
        kaq=kaq,
        c=c_polder,
        name="polder",
        topboundary="leaky",
        hstar=-1.0,
    )
    mlss.solve(silent=True)
    mltr = tf.transient.ModelXsection(naq=1, tmin=1e-5, tmax=1e2)
    tf.transient.XsectionMaq(
        mltr,
        x1=-np.inf,
        x2=0.0,
        z=z_riv,
        kaq=kaq,
        c=c_riv,
        Saq=Saq,
        name="river",
        topboundary="semi",
        tsandhstar=tsandhstar,
    )
    tf.transient.XsectionMaq(
        mltr,
        x1=0.0,
        x2=np.inf,
        z=z_polder,
        kaq=kaq,
        c=c_polder,
        Saq=Saq,
        name="polder",
        topboundary="semi",
    )
    mltr.solve(silent=True)
    cal = tf.Calibrate(transient_model=mltr, steady_model=mlss)
    obslist = [obs1, obs2, obs3, obs4]
    xobs = [0.0, 54.0, 172.0, 194.0]
    for iobs, ix in zip(obslist, xobs):
        t = (iobs.index - river.index[0]).total_seconds() / (3600 * 24)
        cal.add_head_time_series(iobs.name, x=ix, y=0.0, layer=0, t=t, h=iobs.values)
    return cal

In [ ]:
# your code here

So what is the solution to this problem, and how should we modify our calibration?

Well, there isn't sufficient information in the model to estimate each parameter
individually. We can really only estimate the ratio between $k$ and $S_s$ (also known
as aquifer diffusivity) and the ratio of the leakage factors $\lambda =\sqrt{kHc}$
between the river and the polder.

**Bonus exercise** Show that the above is the case by building models where $k$ and
$S_s$ are multiplied by 2 and the resistances $c$ are divided by 2. The result should
be identical to your initial model.

In [ ]:
# bonus exercise here

So for the calibration in timflow, we have to drop one of the parameters and set it to
our best guess. We have to add other information to constrain one of the
parameters. Then we can estimate the other parameters without running into correlation
issues that can cause our optimization to be highly dependent on our initial guesses.

Note that the resulting fit is still identical, and the aquifer diffusivity and the
ratio of the leakage factors will still be the same as before, but now we can estimate
the values of $S_s$, $c_{\text{riv}}$ and $c_{\text{polder}}$ given the value of $k_h$.